In [1]:
# USAGE
# python detect_mask_image.py --image examples/example_01.png

# import the necessary packages
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import load_model
import numpy as np
import argparse
import cv2
import os
import glob

In [2]:
# construct the argument parser and parse the arguments
ap = argparse.ArgumentParser()
ap.add_argument("-i", "--image", required=True,
	help="path to input image")
ap.add_argument("-f", "--face", type=str,
	default="face_detector",
	help="path to face detector model directory")
ap.add_argument("-m", "--model", type=str,
	default="mask_detector.model",
	help="path to trained face mask detector model")
ap.add_argument("-c", "--confidence", type=float, default=0.5,
	help="minimum probability to filter weak detections")
args = vars(ap.parse_args())

usage: ipykernel_launcher.py [-h] -i IMAGE [-f FACE] [-m MODEL] [-c CONFIDENCE]
ipykernel_launcher.py: error: the following arguments are required: -i/--image


SystemExit: 2

C:\Anaconda\envs\aiml\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [3]:
#import os

#print(os.getcwd())#get current work directory
#print(os.listdir())#see the files and folders inside a particular folder.

In [4]:
# load our serialized face detector model from disk
print("[INFO] loading face detector model...")
prototxtPath = os.path.sep.join(["face_detector", "deploy.prototxt"])#predefined neural network
weightsPath = os.path.sep.join(["face_detector",
	"res10_300x300_ssd_iter_140000.caffemodel"])# weight of pre-learned
net = cv2.dnn.readNet(prototxtPath, weightsPath)#cv2 helps Python work with images and videos.

[INFO] loading face detector model...


In [5]:
print("[INFO] loading face mask detector model...")
model = load_model("mask_detector1.h5")

[INFO] loading face mask detector model...


In [6]:
images = glob.glob("dataset/with_mask/*")+ glob.glob("dataset/without_mask/*")#is a Python module used to find files that match a pattern.

for imagepath in images:

    image = cv2.imread(imagepath)

    if image is None:
        continue

    orig = image.copy()
    (h, w) = image.shape[:2]



In [7]:
# construct a blob from the image, prepares the original image in the standard format required by the pre-trained face-detection model before giving it to the model.
blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300),
	(104.0, 177.0, 123.0))# These are the preoprocessing values for the three channels BGR value for OPen CV others like RGB

In [8]:
blob

array([[[[ 136.,  133.,  132., ...,  120.,  123.,  117.],
         [ 127.,  130.,  134., ...,  131.,  130.,  122.],
         [ 120.,  127.,  135., ...,  145.,  139.,  128.],
         ...,
         [ -80.,  -81.,  -79., ...,  133.,  140.,  107.],
         [ -77.,  -78.,  -81., ...,   99.,  134.,  115.],
         [ -76.,  -75.,  -84., ...,   65.,  127.,  126.]],

        [[  65.,   61.,   59., ...,   16.,    9.,    3.],
         [  56.,   58.,   61., ...,   31.,   21.,   12.],
         [  49.,   56.,   62., ...,   50.,   37.,   25.],
         ...,
         [-152., -151., -146., ...,   65.,   70.,   42.],
         [-153., -153., -151., ...,   27.,   61.,   46.],
         [-154., -152., -156., ...,   -8.,   54.,   55.]],

        [[ 120.,  118.,  120., ...,   65.,   60.,   54.],
         [ 111.,  115.,  121., ...,   82.,   74.,   65.],
         [ 104.,  111.,  122., ...,  105.,   95.,   81.],
         ...,
         [-109., -109., -105., ...,  109.,  112.,   77.],
         [-113., -112., -1

In [9]:
# pass the blob through the network and obtain the face detections
net.setInput(blob)
detections = net.forward()

In [10]:
detections.shape#200images

(1, 1, 200, 7)

In [17]:
CONFIDENCE=0.5
# loop over the detections
for i in range (0,detections.shape[2]):# Looping all possible 
    # extract the confidence (i.e., probability) associated with
	# the detection
    confidence=detections[0,0,i,2]
    # filter out weak detections by ensuring the confidence is
	# greater than the minimum confidence
    
    if confidence > CONFIDENCE:
        # compute the (x, y)-coordinates of the bounding box for
		# the object
        box=detections[0,0,i,3:7]*np.array([w,h,w,h])
        (startX,startY,endX,endY)=box.astype("int")
        # ensure the bounding boxes fall within the dimensions of
		# the frame
        (startX,startY)=(max(0,startX),max(0,startY))
        (endX,endY)=(min(w-1,endX),min(h-1,endY))
        # extract the face ROI, convert it from BGR to RGB channel
		# ordering, resize it to 224x224, and preprocess it
        face=image[startY:endY,startX:endX]
        face=cv2.cvtColor(face,cv2.COLOR_BGR2RGB)
        face=cv2.resize(face,(224,224))
        face=img_to_array(face)
        face=preprocess_input(face)
        face=np.expand_dims(face,axis=0)
        # pass the face through the model to determine if the face
		# has a mask or not
        (mask,withoutMask)=model.predict(face)[0]
        # determine the class label and color we'll use to draw
		# the bounding box and text
        label="Mask" if mask > withoutMask else "No Mask"
        color=(0,250,0) if label=="Mask" else (0,0,255)
        # include the probability in the label
        label="{}:{:.2f}%".format(label,max(mask,withoutMask)*100)
        # display the label and bounding box rectangle on the output
		# frame
        cv2.putText(image,label,(startX,startY-10),
            cv2.FONT_HERSHEY_SIMPLEX,0.45,color,2)
        cv2.rectangle(image,(startX,startY),(endX,endY),color,2)
#show the output image
cv2.imshow("Output",image)
cv2.waitKey(0)
cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step


In [12]:
box

array([ 98.05503839,  24.68453729, 167.63441461, 141.83260655])